In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 

- 설문 데이터를 불러온다.
- 서로 다른 구조를 가진 2개의 설문 파일(A, B)을 각각 로드한다.
- 이후 분석은 두 데이터를 통합한 마스터 테이블 기준으로 수행한다.

In [6]:
s_df = pd.read_excel("../data/survey_A_mock.xlsx")
y_df = pd.read_excel("../data/survey_B_mock.xlsx")

print("Survey A:", s_df.shape)
print("Survey B:", y_df.shape)

Survey A: (750, 14)
Survey B: (250, 13)


# 2. 컬럼 그룹 정의

- 컬럼을 역할 기준으로 그룹화한다.

    - 기본 정보 컬럼 → 응답 식별 / 세그먼트 분석용
    - 만족도 컬럼 → 리커트 척도 기반 정량 분석용
    - 주관식 컬럼 → 정성 분석용

- 컬럼을 그룹 단위로 관리하면 이후 파생 변수 생성, 필터링, 통계 처리 시 조작 및 확장이 용이하다.

In [ ]:
# 기본 정보 컬럼
base_cols = [
    "response_id",
    "source",
    "cat_01",   # 이용 목적 (usage_type)
    "cat_02",
    "cat_03",
    "cat_06"
]

# 만족도 컬럼
score_cols = [
    "sat_01",
    "sat_02",
    "sat_03",
    "sat_04",
    "sat_05",
    "sat_06"
]

# 주관식 컬럼
text_cols = [
    "text_01"
]

master_cols = base_cols + score_cols + text_cols

# 3. 누락된 컬럼의 데이터값 자동 생성

- 설문 A와 B는 테이블 구조가 완전히 동일하지 않다.
- 기준 스키마(master_cols)에 존재하지만 특정 설문에는 없는 컬럼이 발생할 수 있다.
- 누락된 컬럼은 NaN으로 생성하여 스키마 구조를 정렬한다.

In [8]:
for col in master_cols:
    if col not in s_df.columns:
        s_df[col] = np.nan

for col in master_cols:
    if col not in y_df.columns:
        y_df[col] = np.nan

# 4. 컬럼 정렬

In [9]:
s_df = s_df[master_cols]
y_df = y_df[master_cols]

# 5. Master Table 생성

- 정렬된 설문 A/B 데이터를 병합하여 최종 마스터 테이블을 생성한다.
- 이후 분석을 위한 파생 변수 생성:
    - 저만족 플래그 (low_sat_flag) <br/>
        → 만족도 문항 중 1개라도 3점 이하 선택 시 True
    - 심각 저만족 플래그 (serious_sat_flag) <br/>
        → 만족도 문항 중 1개라도 2점 이하 선택 시 True
- 플래그 기반 비율 지표를 계산하여 전체 불만족 구조를 확인한다.

In [10]:
master_df = pd.concat([s_df, y_df], ignore_index=True)

print("Master:", master_df.shape)

Master: (1000, 13)


# 6. 저만족 / 심각 플래그

In [ ]:
master_df['low_sat_flag'] = (master_df[score_cols] <= 3).any(axis=1)
master_df['serious_sat_flag'] = (master_df[score_cols] <= 2).any(axis=1)

In [13]:
print("저만족 비율:", master_df['low_sat_flag'].mean())
print("심각 저만족 비율:", master_df['serious_sat_flag'].mean())

저만족 비율: 0.377
심각 저만족 비율: 0.12


# 7. 파생 지표

- 설문 구조 불일치로 인해 일부 세부 문항은 평균 기반 파생 지표로 변환하여 기준 스키마에 정렬하였다.

In [14]:
master_df["idx_01"] = master_df["sat_02"]
master_df["idx_02"] = master_df["sat_03"]
master_df["idx_03"] = master_df[["sat_04","sat_05","sat_06"]].mean(axis=1)

In [16]:
print("중복:", master_df.duplicated(subset=["source","response_id"]).sum())
print("저만족 비율:", master_df['low_sat_flag'].mean())
print("심각 비율:", master_df['serious_sat_flag'].mean())

중복: 0
저만족 비율: 0.377
심각 비율: 0.12


# 8. 저장

In [18]:
master_df.to_excel("../data/master_table.xlsx", index=False)
master_df.to_parquet("../data/master_table.parquet", index=False)